# 01 - CV Transformer Training and 
CV Extraction


In [ ]:
import sys
from pathlib import Path

SRC_DIR = (Path.cwd().resolve() / '..' / 'src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Hard-reload local package modules so notebook always sees latest edits.
for name in list(sys.modules.keys()):
    if name == 'course_project' or name.startswith('course_project.'):
        del sys.modules[name]

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from graph_utils import calc_p_ratio_box

from course_project.config import ExperimentConfig
from course_project.runner import run_experiment
from course_project.data import load_dataset
from course_project.graph import build_graph
from course_project.hessian import collect_cv_lambda5_points
from course_project.models import create_model, resolve_model_inputs



In [ ]:
import random
import shutil

seed = 1
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

fixed_params = {
    'K1': 160,
    'hidden_size': 160,
    'learning_rate': 1e-4,
}

time_lag_steps = 0
time_lag_weight = 0

full_dataset_path = '../data/2340_dePablo_networks_OOL_undirected.pt'

# Data split parameters
train_count = 2
val_count = 15

all_sims = load_dataset(full_dataset_path)
train_sims = all_sims[:train_count]
val_sims = all_sims[train_count:train_count + val_count]
test_sims = all_sims[train_count + val_count:]

print('source dataset:', full_dataset_path)
print('train sims:', len(train_sims), 'val sims:', len(val_sims), 'test sims:', len(test_sims))

common_cfg = dict(
    dataset_path=full_dataset_path,
    train_count=train_count,
    val_count=val_count,
    output_root='../results',
    device='cuda',
    pos_dim=2,
    history=1,
    limit=100,
    n_layers=2,
    learning_rate_decay=0.997,
    epochs=200,
    val_every=20,
    cv_eval_every=20,
    freeze_normalizers_after_epoch=10,
)

cfg_cv_transformer = ExperimentConfig(
    run_name='cv_transformer_single',
    model_type='cv_transformer',
    seed=seed,
    hidden_size=int(fixed_params['hidden_size']),
    learning_rate=float(fixed_params['learning_rate']),
    model_extras={
        'num_mlp': 3,
        'K1': int(fixed_params['K1']),
        'CV': 2,
        'transformer_layers': 1,
        'transformer_heads': 1,
        'transformer_dropout': 0.001,
        'edge_aggr': 'mean',
        'use_local_skip': False,
        'time_lag_steps': int(time_lag_steps),
        'time_lag_weight': float(time_lag_weight),
    },
    **common_cfg,
)

metrics = run_experiment(cfg_cv_transformer)

run_dir = Path(cfg_cv_transformer.output_root) / cfg_cv_transformer.run_name
stats = torch.load(run_dir / 'train_stats.pt', map_location='cpu', weights_only=False)
epochs = np.asarray(stats['epoch'], dtype=int)
cv_fit_r2 = np.asarray(stats['cv_fit_r2'], dtype=float)

best_idx = int(np.nanargmax(cv_fit_r2))
best_epoch = int(epochs[best_idx])
best_score = float(cv_fit_r2[best_idx])
best_ckpt = run_dir / 'rollout_checkpoints' / f'epoch_{best_epoch:04d}.pt'

selection_json = Path(common_cfg['output_root']) / 'cv_transformer_best_cv_selection.json'
best_ckpt_out = Path(common_cfg['output_root']) / 'cv_transformer_best_cv_checkpoint.pt'
shutil.copy2(best_ckpt, best_ckpt_out)

selection = {
    'run_name': str(cfg_cv_transformer.run_name),
    'best_cv_fit_r2': best_score,
    'best_cv_epoch': best_epoch,
    'checkpoint_path': str(best_ckpt),
    'saved_checkpoint_path': str(best_ckpt_out),
    'fixed_params': fixed_params,
}
selection_json.write_text(json.dumps(selection, indent=2))

print('single run:', cfg_cv_transformer.run_name)
print('best cv_fit_r2:', best_score, 'epoch:', best_epoch)
print('saved best checkpoint:', best_ckpt_out)
print('saved selection:', selection_json)
print('selected_checkpoint_score:', float(metrics['selected_checkpoint_score']))


In [ ]:
# CV-fit metric over epochs.
run_dir = Path(cfg_cv_transformer.output_root) / cfg_cv_transformer.run_name
stats = torch.load(run_dir / 'train_stats.pt', map_location='cpu', weights_only=False)

epochs = np.asarray(stats['epoch'], dtype=int)
cv_fit_r2 = np.asarray(stats['cv_fit_r2'], dtype=float)

mask_cv = np.isfinite(cv_fit_r2)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(epochs[mask_cv], cv_fit_r2[mask_cv], '-', linewidth=2.0, color='tab:blue')
ax.set_xlabel('epoch')
ax.set_ylabel('cv_fit_r2')
ax.grid(alpha=0.25)
ax.set_title('CV fit over epochs')

plt.tight_layout()
plt.show()

best_i = int(np.nanargmax(cv_fit_r2))
print('best cv_fit_r2=', float(cv_fit_r2[best_i]), 'at epoch', int(epochs[best_i]))


In [ ]:
# Load model from the saved best CV-fit checkpoint.
output_root = Path(cfg_cv_transformer.output_root)
selection = json.loads((output_root / 'cv_transformer_best_cv_selection.json').read_text())

run_dir = output_root / selection['run_name']
cfg_dict = json.loads((run_dir / 'config.json').read_text())
cfg_loaded = ExperimentConfig(**cfg_dict)

device = cfg_loaded.device
if device == 'cuda' and not torch.cuda.is_available():
    device = 'cpu'

all_sims = load_dataset(cfg_loaded.dataset_path)
val_data = all_sims[cfg_loaded.train_count:cfg_loaded.train_count + cfg_loaded.val_count]
test_data = all_sims[cfg_loaded.train_count + cfg_loaded.val_count:]

init_frames = [val_data[0][i].to(device) for i in range(cfg_loaded.history + 1)]
init_graph = build_graph(input_graphs=init_frames).to(device)

model = create_model(
    model_type=cfg_loaded.model_type,
    init_graph=init_graph,
    pos_dim=cfg_loaded.pos_dim,
    hidden_size=cfg_loaded.hidden_size,
    n_layers=cfg_loaded.n_layers,
    extras=cfg_loaded.model_extras,
).to(device)

best_ckpt = Path(selection['saved_checkpoint_path'])
model.load_checkpoint(str(best_ckpt))

model.eval()
if hasattr(model, 'freeze_normalizers'):
    model.freeze_normalizers = True
for p in model.parameters():
    p.requires_grad = False
model_inputs_cls = resolve_model_inputs(cfg_loaded.model_type)

print('Loaded model:', cfg_loaded.model_type, 'device=', device)
print('Run dir:', run_dir)
print('Loaded best CV checkpoint:', best_ckpt)
print('best cv_fit_r2:', float(selection['best_cv_fit_r2']), 'epoch:', int(selection['best_cv_epoch']))
print('effective num CVs (CV):', cfg_loaded.model_extras['CV'])
print('validation sims:', len(val_data), '| test sims:', len(test_data))


In [ ]:
# Extract trajectory-level CV summaries and final p-ratio for VAL and TEST.
def extract_cv_summary(sims, split_name):
    rows = []
    with torch.no_grad():
        for sim_idx, sim in enumerate(sims):
            n_local = len(sim) - 1

            final_pr = float(calc_p_ratio_box(sim, -1))

            cv_values = []
            for t in range(cfg_loaded.history, n_local):
                frames = [sim[i].to(device) for i in range(t - cfg_loaded.history, t + 1)]
                if t > 0:
                    prev_pos = sim[t - 1].to(device).x[:, : cfg_loaded.pos_dim]
                    cur_pos = frames[-1].x[:, : cfg_loaded.pos_dim]
                    frames[-1].vel_state = cur_pos - prev_pos

                input_graph = build_graph(frames).to(device)
                cv = model.extract_cv(input_graph, is_training=False).squeeze(0).detach().cpu().numpy()
                cv = np.asarray(cv, dtype=float).reshape(-1)
                cv_values.append(cv)

            cv_mean = np.mean(np.stack(cv_values, axis=0), axis=0)
            row = {
                'split': split_name,
                'sim_idx': sim_idx,
                'num_frames_used': len(cv_values),
                'final_p_ratio': final_pr,
            }
            for i, value in enumerate(cv_mean, start=1):
                row[f'cv_{i}'] = float(value)
            rows.append(row)
    return pd.DataFrame(rows).sort_values('sim_idx').reset_index(drop=True)


df_val = extract_cv_summary(val_data, 'val')
df_test = extract_cv_summary(test_data, 'test')

cv_cols = [c for c in df_val.columns if c.startswith('cv_')]

val_corr = {}
for col in cv_cols:
    x = df_val[col].to_numpy(dtype=float)
    y = df_val['final_p_ratio'].to_numpy(dtype=float)
    val_corr[col] = float(np.corrcoef(x, y)[0, 1])

best_cv_col = max(cv_cols, key=lambda c: abs(val_corr[c]))
best_cv_idx = int(best_cv_col.split('_')[1]) - 1

print('best CV on val:', best_cv_col, '(index:', best_cv_idx, ')')
print('val pearson:', val_corr[best_cv_col])
print('test pearson:', float(np.corrcoef(df_test[best_cv_col].to_numpy(dtype=float), df_test['final_p_ratio'].to_numpy(dtype=float))[0, 1]))


In [ ]:
# Scatter of best CV vs final p-ratio on VAL and TEST (same selected checkpoint and CV).
x_val = df_val[best_cv_col].to_numpy(dtype=float)
y_val = df_val['final_p_ratio'].to_numpy(dtype=float)
r_val = float(np.corrcoef(x_val, y_val)[0, 1])

x_test = df_test[best_cv_col].to_numpy(dtype=float)
y_test = df_test['final_p_ratio'].to_numpy(dtype=float)
r_test = float(np.corrcoef(x_test, y_test)[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))

axes[0].scatter(x_val, y_val, s=28, alpha=0.75)
axes[0].set_title(f'VAL: {best_cv_col} vs final p-ratio (r={r_val:.3f})')
axes[0].set_xlabel(best_cv_col)
axes[0].set_ylabel('final_p_ratio')
axes[0].grid(alpha=0.25)

axes[1].scatter(x_test, y_test, s=20, alpha=0.65)
axes[1].set_title(f'TEST: {best_cv_col} vs final p-ratio (r={r_test:.3f})')
axes[1].set_xlabel(best_cv_col)
axes[1].set_ylabel('final_p_ratio')
axes[1].grid(alpha=0.25)

fig.suptitle(f"{cfg_loaded.run_name}: best CV selected on VAL, evaluated on TEST", y=1.02)
fig.tight_layout()

val_csv = run_dir / 'cv_transformer_val_best_cv_vs_pratio.csv'
test_csv = run_dir / 'cv_transformer_test_best_cv_vs_pratio.csv'
plot_png = run_dir / 'cv_transformer_best_cv_val_test_scatter.png'

df_val[['sim_idx', best_cv_col, 'final_p_ratio']].to_csv(val_csv, index=False)
df_test[['sim_idx', best_cv_col, 'final_p_ratio']].to_csv(test_csv, index=False)
fig.savefig(plot_png, dpi=180, bbox_inches='tight')

plt.show()

print('best checkpoint epoch:', int(selection['best_cv_epoch']))
print('saved:', val_csv)
print('saved:', test_csv)
print('saved:', plot_png)


In [ ]:
# Hessian check at the end: frame-level best CV vs lambda5 on chosen split.
hessian_split = 'test'  # 'val' or 'test'
max_hessian_sims = 10

sims_hessian = val_data if hessian_split == 'val' else test_data

pts = collect_cv_lambda5_points(
    model,
    sims_hessian,
    history=cfg_loaded.history,
    pos_dim=cfg_loaded.pos_dim,
    device=device,
    max_sims=max_hessian_sims,
    cv_index=best_cv_idx,
)

x = pts['cv']
y = pts['lambda5']

fig, ax = plt.subplots(figsize=(7.2, 5.0))
ax.scatter(x, y, s=8, alpha=0.35)
ax.set_xlabel(best_cv_col)
ax.set_ylabel('lambda5')
ax.set_title(f'Frame-level {best_cv_col} vs lambda5 ({hessian_split}, up to {max_hessian_sims} sims)')
ax.grid(alpha=0.25)
fig.tight_layout()

hessian_png = run_dir / f'cv_transformer_{hessian_split}_best_cv_vs_lambda5.png'
fig.savefig(hessian_png, dpi=180, bbox_inches='tight')
plt.show()

pearson = float(np.corrcoef(x, y)[0, 1])
print('hessian split:', hessian_split)
print('points:', len(x))
print(f'pearson({best_cv_col}, lambda5)=', pearson)
print('saved:', hessian_png)
